# Kafka

**Objective:** Publish and consume a versioned LLM job with durable producer settings, explicit offsets, and testable adapters.

## Simple version

A producer appends events to a topic. Consumers read them independently; the offset records each consumer group's progress.

In [1]:
topic: list[dict[str, str]] = []
topic.append({"key": "job-1", "value": "Explain Kafka"})

offset = 0
event = topic[offset]
print(event)
offset += 1
print("Committed offset:", offset)

{'key': 'job-1', 'value': 'Explain Kafka'}
Committed offset: 1


```mermaid
flowchart LR
    API[API producer] --> Topic[Topic partitions]
    Topic --> Workers[Worker consumer group]
    Topic --> Analytics[Analytics consumer group]
    Workers -->|commit after success| Offsets[(Offsets)]
```

The record key selects a partition. Ordering is guaranteed within that partition, not across the whole topic.

## Polished version

The domain owns the message contract. The Kafka adapter owns bytes, headers, topics, offsets, client errors, and lifecycle.

In [2]:
from datetime import UTC, datetime
from typing import Literal, Protocol
from uuid import UUID

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    SecretStr,
    ValidationError,
    model_validator,
)


class InvalidBrokerMessage(ValueError):
    pass


class BrokerUnavailable(RuntimeError):
    pass


class ChatMessageV1(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=20_000)


class GenerateChatV1(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    event_type: Literal["chat.generate.requested"] = "chat.generate.requested"
    schema_version: Literal[1] = 1
    event_id: UUID
    job_id: UUID
    occurred_at: datetime
    messages: tuple[ChatMessageV1, ...] = Field(min_length=1, max_length=50)
    model: str | None = Field(default=None, min_length=1, max_length=200)
    temperature: float = Field(default=0.2, ge=0, le=2)

    @model_validator(mode="after")
    def validate_total_size(self) -> "GenerateChatV1":
        if sum(len(message.content) for message in self.messages) > 100_000:
            raise ValueError("Messages exceed 100000 characters")
        return self

    def encode(self) -> bytes:
        return self.model_dump_json().encode()

    @classmethod
    def decode(cls, value: bytes) -> "GenerateChatV1":
        try:
            return cls.model_validate_json(value)
        except ValidationError as error:
            raise InvalidBrokerMessage("Invalid GenerateChatV1 payload") from error


job = GenerateChatV1(
    event_id=UUID("00000000-0000-0000-0000-000000000001"),
    job_id=UUID("00000000-0000-0000-0000-000000000002"),
    occurred_at=datetime(2026, 9, 1, tzinfo=UTC),
    messages=(ChatMessageV1(role="user", content="Explain Kafka"),),
)
print(GenerateChatV1.decode(job.encode()).job_id)

00000000-0000-0000-0000-000000000002


### Producer and consumer adapters

The producer waits for acknowledgements from all in-sync replicas and enables idempotent production. The consumer disables automatic commits and advances one partition only after processing or dead-lettering succeeds.

In [3]:
from collections.abc import Mapping
from dataclasses import dataclass, field
from typing import Any

from aiokafka import AIOKafkaConsumer, AIOKafkaProducer, TopicPartition
from aiokafka.errors import KafkaError


@dataclass(frozen=True, slots=True)
class KafkaSettings:
    bootstrap_servers: str
    topic: str
    dead_letter_topic: str
    group_id: str
    client_id: str = "llm-api"
    request_timeout_ms: int = 30_000
    security_protocol: Literal["PLAINTEXT", "SSL", "SASL_PLAINTEXT", "SASL_SSL"] = (
        "PLAINTEXT"
    )
    sasl_mechanism: Literal["PLAIN", "SCRAM-SHA-256", "SCRAM-SHA-512"] = "PLAIN"
    sasl_username: str | None = None
    sasl_password: SecretStr | None = field(default=None, repr=False)

    def client_options(self) -> dict[str, object]:
        return {
            "bootstrap_servers": self.bootstrap_servers,
            "security_protocol": self.security_protocol,
            "sasl_mechanism": self.sasl_mechanism,
            "sasl_plain_username": self.sasl_username,
            "sasl_plain_password": (
                self.sasl_password.get_secret_value() if self.sasl_password else None
            ),
            "request_timeout_ms": self.request_timeout_ms,
        }


class KafkaProducerClient(Protocol):
    async def send_and_wait(
        self,
        topic: str,
        value: bytes,
        *,
        key: bytes | None = None,
        headers: list[tuple[str, bytes]] | None = None,
    ) -> object: ...


class KafkaRecord(Protocol):
    topic: str
    partition: int
    offset: int
    key: bytes | None
    value: bytes


class KafkaConsumerClient(Protocol):
    async def getone(self) -> KafkaRecord: ...

    async def commit(self, offsets: Mapping[TopicPartition, int]) -> None: ...


class JobHandler(Protocol):
    async def handle(self, event: GenerateChatV1) -> None: ...


class KafkaJobPublisher:
    def __init__(self, producer: KafkaProducerClient, settings: KafkaSettings) -> None:
        self.producer = producer
        self.settings = settings

    async def publish(self, event: GenerateChatV1) -> None:
        try:
            await self.producer.send_and_wait(
                self.settings.topic,
                event.encode(),
                key=str(event.job_id).encode(),
                headers=[
                    ("event_type", event.event_type.encode()),
                    ("schema_version", str(event.schema_version).encode()),
                ],
            )
        except KafkaError as error:
            raise BrokerUnavailable("Kafka publish failed") from error

    async def dead_letter(self, record: KafkaRecord, reason: str) -> None:
        try:
            await self.producer.send_and_wait(
                self.settings.dead_letter_topic,
                record.value,
                key=record.key,
                headers=[("failure_reason", reason[:500].encode())],
            )
        except KafkaError as error:
            raise BrokerUnavailable("Kafka dead-letter publish failed") from error


class KafkaWorker:
    def __init__(
        self,
        consumer: KafkaConsumerClient,
        publisher: KafkaJobPublisher,
        handler: JobHandler,
    ) -> None:
        self.consumer = consumer
        self.publisher = publisher
        self.handler = handler

    async def run_once(self) -> None:
        try:
            record = await self.consumer.getone()
        except KafkaError as error:
            raise BrokerUnavailable("Kafka consume failed") from error

        try:
            event = GenerateChatV1.decode(record.value)
        except InvalidBrokerMessage as error:
            # Commit only after the poison message is safely copied to the DLQ.
            await self.publisher.dead_letter(record, str(error))
            await self._commit(record)
            return

        # A handler exception leaves the offset uncommitted for redelivery.
        await self.handler.handle(event)
        await self._commit(record)

    async def run_forever(self) -> None:
        while True:
            await self.run_once()

    async def _commit(self, record: KafkaRecord) -> None:
        partition = TopicPartition(record.topic, record.partition)
        try:
            await self.consumer.commit({partition: record.offset + 1})
        except KafkaError as error:
            raise BrokerUnavailable("Kafka offset commit failed") from error


def build_kafka_clients(
    settings: KafkaSettings,
) -> tuple[AIOKafkaProducer, AIOKafkaConsumer]:
    options = settings.client_options()
    producer = AIOKafkaProducer(
        **options,
        client_id=f"{settings.client_id}-producer",
        acks="all",
        enable_idempotence=True,
    )
    consumer = AIOKafkaConsumer(
        settings.topic,
        **options,
        client_id=f"{settings.client_id}-consumer",
        group_id=settings.group_id,
        enable_auto_commit=False,
        auto_offset_reset="earliest",
    )
    return producer, consumer

### Lifecycle and broker-free adapter test

The production context starts both clients and closes them in reverse order. The test replaces only the network clients.

In [4]:
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager


@asynccontextmanager
async def kafka_runtime(
    settings: KafkaSettings,
    handler: JobHandler,
) -> AsyncIterator[tuple[KafkaJobPublisher, KafkaWorker]]:
    producer, consumer = build_kafka_clients(settings)
    await producer.start()
    try:
        await consumer.start()
        try:
            publisher = KafkaJobPublisher(producer, settings)
            yield publisher, KafkaWorker(consumer, publisher, handler)
        finally:
            await consumer.stop()
    finally:
        await producer.stop()


class FakeKafkaProducer:
    def __init__(self) -> None:
        self.records: list[dict[str, Any]] = []

    async def send_and_wait(
        self,
        topic: str,
        value: bytes,
        *,
        key: bytes | None = None,
        headers: list[tuple[str, bytes]] | None = None,
    ) -> object:
        self.records.append(
            {"topic": topic, "value": value, "key": key, "headers": headers}
        )
        return object()


@dataclass(frozen=True)
class FakeKafkaRecord:
    topic: str
    partition: int
    offset: int
    key: bytes | None
    value: bytes


class FakeKafkaConsumer:
    def __init__(self, record: FakeKafkaRecord) -> None:
        self.record = record
        self.commits: list[Mapping[TopicPartition, int]] = []

    async def getone(self) -> FakeKafkaRecord:
        return self.record

    async def commit(self, offsets: Mapping[TopicPartition, int]) -> None:
        self.commits.append(offsets)


class RecordingHandler:
    def __init__(self) -> None:
        self.jobs: list[UUID] = []

    async def handle(self, event: GenerateChatV1) -> None:
        self.jobs.append(event.job_id)


settings = KafkaSettings(
    bootstrap_servers="localhost:9092",
    topic="chat.generate.requested.v1",
    dead_letter_topic="chat.generate.requested.dlq.v1",
    group_id="llm-workers-v1",
)
producer = FakeKafkaProducer()
publisher = KafkaJobPublisher(producer, settings)
await publisher.publish(job)

record = FakeKafkaRecord(
    topic=settings.topic,
    partition=0,
    offset=7,
    key=str(job.job_id).encode(),
    value=job.encode(),
)
consumer = FakeKafkaConsumer(record)
handler = RecordingHandler()
await KafkaWorker(consumer, publisher, handler).run_once()

assert handler.jobs == [job.job_id]
assert list(consumer.commits[0].values()) == [8]

# Invalid payloads move to the DLQ before their source offset is committed.
poison = FakeKafkaRecord(settings.topic, 0, 8, b"job-2", b"not-json")
poison_consumer = FakeKafkaConsumer(poison)
await KafkaWorker(poison_consumer, publisher, handler).run_once()
assert producer.records[-1]["topic"] == settings.dead_letter_topic
assert list(poison_consumer.commits[0].values()) == [9]
print(producer.records[0]["topic"], consumer.commits[0], "DLQ verified")

chat.generate.requested.v1 {TopicPartition(topic='chat.generate.requested.v1', partition=0): 8} DLQ verified


### Optional durability lab

This proves that an acknowledged event survives broker-container recreation:

1. Run `make brokers-up` in the repository root.
2. Set `KAFKA_DURABILITY_STEP = "publish"` and run the cell.
3. Run `make brokers-recreate`; the container is replaced but its named volume remains.
4. Set the step to `"consume"` and rerun the cell.
5. Run `make brokers-down` when finished.

This single-broker lab teaches disk persistence. Production durability also needs replicated partitions across multiple brokers.

In [5]:
import asyncio


KAFKA_DURABILITY_STEP = ""  # Change to "publish", then "consume".
lab_settings = KafkaSettings(
    bootstrap_servers="localhost:9092",
    topic="chat.generate.durability.v1",
    dead_letter_topic="chat.generate.requested.dlq.v1",
    group_id="kafka-durability-lab-v1",
)
lab_handler = RecordingHandler()

if KAFKA_DURABILITY_STEP in {"publish", "consume"}:
    async with kafka_runtime(lab_settings, lab_handler) as (publisher, worker):
        if KAFKA_DURABILITY_STEP == "publish":
            await publisher.publish(job)
            print("Published and acknowledged. Recreate Kafka next.")
        else:
            async with asyncio.timeout(20):
                await worker.run_once()
            assert lab_handler.jobs == [job.job_id]
            print("Consumed after recreation:", lab_handler.jobs[0])
else:
    print("Durability lab skipped; follow the steps above to run it.")

Durability lab skipped; follow the steps above to run it.


### Production rules

- Provision topics, partition count, replication, retention, and `min.insync.replicas` outside application code.
- Store credentials in deployment configuration and use TLS/SASL outside local development.
- Make the handler idempotent with `event_id`; an offset commit can fail after the side effect succeeds.
- Monitor consumer lag, rebalance frequency, dead-letter volume, publish latency, and commit failures.
- Evolve schemas compatibly and keep old consumers readable during rolling deployments.

## Applied in this repository

The LLM project currently uses the broker-independent [JobQueue port](../00P2-project-llm-api/app/application/ports.py) with a [Redis/ARQ adapter](../00P2-project-llm-api/app/infrastructure/queue.py). `KafkaJobPublisher` and `KafkaWorker` demonstrate alternative adapters; they are not wired into the project. Job-status results would still need durable result storage.